In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn as nn

## Data Engineering

In [ ]:
df = pd.read_csv('C:\\Users\\alvar\\Documents\\torch_templates\\DATA\\NYCTaxiFares.csv')
df.head()

In [ ]:
df['fare_class'].value_counts() # fare < 10 = 0, fare >= 10 = 1

In [ ]:
def haversine_distance(df, lat1, long1, lat2, long2):
    r = 6371
    phi1 = np.radians(df[lat1])
    phi2 = np.radians(df[lat2])
    delta_phi = np.radians(df[lat2] - df[lat1])
    delta_lambda = np.radians(df[long2] - df[long1])
    a = np.sin(delta_phi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    d = (r * c)

    return d

In [ ]:
df['distance'] = haversine_distance(df, 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime']) - pd.Timedelta(hours=4) # saving time

df['Hour'] = df['pickup_datetime'].dt.hour
df['Weekday'] = df['pickup_datetime'].dt.strftime("%a")
df

## Separate Categorical and Continuous

In [ ]:
df.columns

In [ ]:
cat_cols = ['Hour', 'Weekday']
cont_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'distance']
y_col = ['fare_class']

In [ ]:
for cat in cat_cols:
    df[cat] = df[cat].astype('category')

### Categorify

In [ ]:
cats = np.stack([df[col].cat.codes.values for col in cat_cols], 1)
cats = torch.tensor(cats, dtype=torch.int64)

In [ ]:
conts = np.stack([df[col].values for col in cont_cols], 1)
conts = torch.tensor(conts, dtype=torch.float)

In [ ]:
y = torch.tensor(df[y_col].values).flatten()

## Set and Embedding Size

In [ ]:
cat_szs = [len(df[col].cat.categories) for col in cat_cols]
emb_szs = [(size, min(50, (size+1)//2)) for size in cat_szs]

In [ ]:
emb_szs

## Creating the Model

In [ ]:
class TabularModel(nn.Module):

    def __init__(self, emb_szs, n_cont, out_sz, layers, p):
        super().__init__()
        self.embeds = nn.ModuleList([nn.Embedding(ni, nf) for ni, nf in emb_szs])
        self.emb_drop = nn.Dropout(p)
        self.bn_count = nn.BatchNorm1d(n_cont)

        layerlist = []
        n_emb = sum((nf for ni, nf in emb_szs))
        n_in = n_emb + n_cont

        for layer in layers:
            layerlist.append(nn.Linear(n_in, layer))
            layerlist.append(nn.ReLU(inplace=True))
            layerlist.append(nn.BatchNorm1d(layer))
            layerlist.append(nn.Dropout(p))
            n_in = layer

        layerlist.append(nn.Linear(layers[-1], out_sz))

        self.layers = nn.Sequential(*layerlist)

    def forward(self, x_cat, x_cont):
        embeddings = []
        for i, e in enumerate(self.embeds):
            embeddings.append(e(x_cat[:,i]))
        x = torch.cat(embeddings, 1)
        x = self.emb_drop(x)

        x_cont = self.bn_count(x_cont)
        x = torch.cat([x, x_cont], 1)
        x = self.layers(x)

        return x

In [ ]:
model = TabularModel(emb_szs=emb_szs, n_cont=conts.shape[1], out_sz=2, layers=[200,100], p=0.5)

In [ ]:
model

## Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
batch_size = 60000
test_size = int(batch_size * .2)

cat_train = cats[:batch_size-test_size]
cat_test = cats[batch_size-test_size:batch_size]
cont_train = conts[:batch_size-test_size]
cont_test = conts[batch_size-test_size:batch_size]
y_train = y[:batch_size-test_size]
y_test = y[batch_size-test_size:batch_size]

In [ ]:
epochs = 300

losses = []

for i in range(1, epochs+1):
    y_pred = model(cat_train, cont_train)
    loss = criterion(y_pred, y_train)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'Epoch: {i}, loss = {loss.item()}')

In [ ]:
plt.plot(range(epochs), losses)
plt.xlabel('epoch')
plt.xlabel('loss');

## Evaluation

In [ ]:
with torch.no_grad():
    y_val = model(cat_test, cont_test)
    loss = criterion(y_val, y_test)
print(f'Cross Entropy Loss: {loss}')

In [ ]:
predictions = [i.argmax().item() for i in y_val]

In [ ]:
from sklearn.metrics import classification_report


print(classification_report(y_test, predictions))

## Loading a Saved Model

In [ ]:
torch.save(model.state_dict(), 'C:\\Users\\alvar\\Documents\\torch_templates\\ANN\\taxi_model.pt')

In [ ]:
emb_szs = [(24, 12), (7, 4)]
new_model = TabularModel(emb_szs=emb_szs, n_cont=6, out_sz=2, layers=[200,100], p=0.5)

In [ ]:
new_model.load_state_dict(torch.load('C:\\Users\\alvar\\Documents\\torch_templates\\ANN\\taxi_model.pt'))
new_model.eval()

## Prediction

In [ ]:
def predict(mdl, pickup_lat, pickup_long, dropoff_lat, dropoff_long, passengers, datetime): # pass in the name of the new model
    # PREPROCESS THE DATA
    dfx_dict = {'pickup_latitude':pickup_lat,'pickup_longitude':pickup_long,'dropoff_latitude':dropoff_lat,
         'dropoff_longitude':dropoff_lat,'passenger_count':passengers,'EDTdate':datetime}
    dfx = pd.DataFrame(dfx_dict, index=[0])
    dfx['dist_km'] = haversine_distance(dfx,'pickup_latitude', 'pickup_longitude',
                                        'dropoff_latitude', 'dropoff_longitude')
    dfx['EDTdate'] = pd.to_datetime(dfx['EDTdate'])
    
    # We can skip the .astype(category) step since our fields are small,
    # and encode them right away
    dfx['Hour'] = dfx['EDTdate'].dt.hour
    dfx['Weekday'] = dfx['EDTdate'].dt.strftime("%a")
    pd.set_option('future.no_silent_downcasting', True)
    dfx['Weekday'] = dfx['Weekday'].replace(['Fri','Mon','Sat','Sun','Thu','Tue','Wed'],
                                            [0,1,2,3,4,5,6]).astype('int64')
    
    # CREATE CAT AND CONT TENSORS
    cat_cols = ['Hour', 'Weekday']
    cont_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude',
                 'dropoff_longitude', 'passenger_count', 'dist_km']
    xcats = np.stack([dfx[col].values for col in cat_cols], 1)
    xcats = torch.tensor(xcats, dtype=torch.int64)
    xconts = np.stack([dfx[col].values for col in cont_cols], 1)
    xconts = torch.tensor(xconts, dtype=torch.float)
    
    # PASS NEW DATA THROUGH THE MODEL WITHOUT PERFORMING A BACKPROP
    with torch.no_grad():
        z = mdl(xcats, xconts).argmax().item()
        
    print(f'\nThe predicted fare class is {z}')

In [ ]:
predict(new_model, 40.5, -73.9, 40.52, -73.92, 2, '2010-05-05 16:50:00')